# 🚬 흡연 분류 AI 해커톤 - V7 (안정화 버전)

## ⚠️ V6 대비 핵심 변경
- ✅ **OOF 기반** 임계값/가중치 결정 (단일 홀드아웃 X)
- ✅ **CatBoost 범주형** 변수 처리 (충치, 단백 등)
- ✅ **파생 피처 최소화** (검증된 핵심만)
- ✅ **이상치 클리핑 OFF**
- ✅ **모델 단순화** (3개: XGB, LGB, CAT)

---

## 📌 STEP 1: 환경 설정

In [ ]:
!pip install -q xgboost lightgbm catboost

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 경로 설정
base_path = '/content/drive/MyDrive/AI_Projects/smoking_hackathon/'
train_path = base_path + 'data/train.csv'
test_path = base_path + 'data/test.csv'
submission_path = base_path + 'data/sample_submission.csv'
result_path = base_path + 'results/'

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import random
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
set_seed(42)

print("✅ 라이브러리 로드 완료!")

## 📌 STEP 2: 데이터 로드 및 컬럼 확인

In [ ]:
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
submission = pd.read_csv(submission_path)

print(f"Train: {train.shape}, Test: {test.shape}")
print(f"\n📋 컬럼명: {train.columns.tolist()}")
print(f"\n📋 데이터 샘플:")
display(train.head(3))

## 📌 STEP 3: 한글 컬럼명 매핑 + 범주형 식별

In [ ]:
def map_and_identify_columns(df):
    """
    한글 컬럼 매핑 + 범주형 컬럼 식별
    """
    df = df.copy()
    
    col_mapping = {}
    categorical_cols = []  # 범주형으로 처리할 컬럼
    
    for col in df.columns:
        col_lower = col.lower()
        
        # ID
        if 'id' in col_lower:
            col_mapping[col] = 'id'
        # 나이
        elif '나이' in col or 'age' in col_lower:
            col_mapping[col] = 'age'
        # 키
        elif '키' in col or 'height' in col_lower:
            col_mapping[col] = 'height'
        # 몸무게
        elif '몸무게' in col or '체중' in col or 'weight' in col_lower:
            col_mapping[col] = 'weight'
        # BMI
        elif 'bmi' in col_lower or '체질량' in col:
            col_mapping[col] = 'bmi'
        # 시력
        elif '시력' in col or 'eyesight' in col_lower:
            col_mapping[col] = 'eyesight'
        # 충치 (⭐ 범주형!)
        elif '충치' in col or 'cavity' in col_lower:
            col_mapping[col] = 'cavity'
            categorical_cols.append('cavity')
        # 공복 혈당
        elif '혈당' in col or '공복' in col:
            col_mapping[col] = 'fasting_blood_sugar'
        # 혈압
        elif '혈압' in col or 'pressure' in col_lower:
            col_mapping[col] = 'blood_pressure'
        # 중성 지방
        elif '중성' in col or 'triglyceride' in col_lower:
            col_mapping[col] = 'triglyceride'
        # 혈청 크레아티닌
        elif '크레' in col or 'creatinine' in col_lower:
            col_mapping[col] = 'serum_creatinine'
        # 콜레스테롤
        elif '콜레스테롤' in col or 'cholesterol' in col_lower:
            col_mapping[col] = 'cholesterol'
        # 고밀도지단백 (HDL)
        elif '고밀도' in col or 'hdl' in col_lower:
            col_mapping[col] = 'hdl'
        # 저밀도지단백 (LDL)
        elif '저밀도' in col or 'ldl' in col_lower:
            col_mapping[col] = 'ldl'
        # 헤모글로빈
        elif '헤모글로빈' in col or 'hemoglobin' in col_lower:
            col_mapping[col] = 'hemoglobin'
        # 단백 (⭐ 범주형!)
        elif '단백' in col and '지단백' not in col:
            col_mapping[col] = 'urine_protein'
            categorical_cols.append('urine_protein')
        # 간 효소율 (GTP)
        elif '간' in col or '효소' in col or 'gtp' in col_lower:
            col_mapping[col] = 'gtp'
        # label
        elif 'label' in col_lower:
            col_mapping[col] = 'label'
        else:
            clean = col.lower().replace(' ', '_').replace('(', '').replace(')', '')
            col_mapping[col] = clean
    
    df = df.rename(columns=col_mapping)
    return df, col_mapping, categorical_cols

# 매핑 적용
train_mapped, col_mapping, cat_cols = map_and_identify_columns(train)
test_mapped, _, _ = map_and_identify_columns(test)

print("📊 컬럼 매핑:")
for orig, eng in col_mapping.items():
    marker = "⭐범주형" if eng in cat_cols else ""
    print(f"  {orig:20} → {eng:20} {marker}")

print(f"\n📋 범주형 컬럼: {cat_cols}")

In [ ]:
# 데이터 분리
train_df = train_mapped.copy()
test_df = test_mapped.copy()

# ID 제거
if 'id' in train_df.columns:
    train_df = train_df.drop('id', axis=1)
    test_df = test_df.drop('id', axis=1)

# X, y 분리
X = train_df.drop('label', axis=1, errors='ignore')
y = train_df['label']
X_test = test_df.drop('label', axis=1, errors='ignore')

print(f"X: {X.shape}, y: {y.shape}, X_test: {X_test.shape}")
print(f"컬럼: {X.columns.tolist()}")
print(f"\n흡연자 비율: {y.mean()*100:.2f}%")

## 📌 STEP 4: ⭐ 피처 엔지니어링 (최소화!)

In [ ]:
def create_minimal_features(df, cat_cols):
    """
    최소한의 검증된 피처만 생성
    - 핵심: hemoglobin, gtp, tg/hdl ratio
    - 과도한 파생 제거
    """
    df = df.copy()
    cols = df.columns.tolist()
    created = []
    
    # 1. TG/HDL 비율 (심혈관 핵심 지표)
    if 'triglyceride' in cols and 'hdl' in cols:
        df['tg_hdl_ratio'] = df['triglyceride'] / (df['hdl'] + 1)
        created.append('tg_hdl_ratio')
    
    # 2. 헤모글로빈 높음 플래그 (흡연자 특성)
    if 'hemoglobin' in cols:
        df['hemo_high'] = (df['hemoglobin'] > 15.5).astype(int)
        created.append('hemo_high')
        cat_cols.append('hemo_high')  # 범주형으로 처리
    
    # 3. GTP 높음 플래그 (흡연자 특성)
    if 'gtp' in cols:
        df['gtp_high'] = (df['gtp'] > 40).astype(int)
        created.append('gtp_high')
        cat_cols.append('gtp_high')  # 범주형으로 처리
    
    # 4. 나이대 (범주형)
    if 'age' in cols:
        df['age_group'] = pd.cut(df['age'], bins=[0,35,45,55,100], labels=[0,1,2,3]).astype(int)
        created.append('age_group')
        cat_cols.append('age_group')  # 범주형으로 처리
    
    # 결측치 처리
    df = df.fillna(0)
    
    print(f"✅ 생성된 파생 피처: {len(created)}개 - {created}")
    return df, cat_cols

# 피처 엔지니어링 적용
print(f"원본 피처 수: {X.shape[1]}개")
X_fe, cat_cols_updated = create_minimal_features(X, cat_cols.copy())
X_test_fe, _ = create_minimal_features(X_test, cat_cols.copy())
print(f"최종 피처 수: {X_fe.shape[1]}개 (파생 {X_fe.shape[1] - X.shape[1]}개 추가)")
print(f"\n📋 범주형 컬럼 (CatBoost용): {cat_cols_updated}")

## 📌 STEP 5: ⭐ 이상치 클리핑 OFF (제거됨)

V6에서 사용한 IQR 클리핑을 **제거**합니다.  
클리핑이 오히려 유용한 신호를 깎을 수 있기 때문입니다.

In [ ]:
# 클리핑 없이 그대로 사용
X_processed = X_fe.copy()
X_test_processed = X_test_fe.copy()

print("✅ 이상치 클리핑 OFF - 원본 데이터 유지")
print(f"   X: {X_processed.shape}")
print(f"   X_test: {X_test_processed.shape}")

## 📌 STEP 6: ⭐ K-Fold OOF 예측 (핵심!)

In [ ]:
print("=" * 60)
print("🎯 K-Fold OOF 예측 (안정적인 검증)")
print("=" * 60)

N_SPLITS = 5
SEED = 42

# 클래스 불균형 비율
scale_pos_weight = (y == 0).sum() / (y == 1).sum()
print(f"클래스 비율: {scale_pos_weight:.2f}")

# CatBoost용 범주형 컬럼명 (DataFrame 사용 시 컬럼명으로 지정)
cat_features_names = [c for c in cat_cols_updated if c in X_processed.columns]
print(f"CatBoost 범주형 컬럼: {cat_features_names}")

# XGBoost, LightGBM용 numpy 변환
X_np = X_processed.values
X_test_np = X_test_processed.values

# CatBoost용 DataFrame 유지 (범주형 처리를 위해)
X_df = X_processed.copy()
X_test_df = X_test_processed.copy()

feature_names = X_processed.columns.tolist()

In [ ]:
# OOF 저장소
oof_xgb = np.zeros(len(X_np))
oof_lgb = np.zeros(len(X_np))
oof_cat = np.zeros(len(X_np))

# Test 예측 저장소
test_xgb = np.zeros(len(X_test_np))
test_lgb = np.zeros(len(X_test_np))
test_cat = np.zeros(len(X_test_np))

# K-Fold
kfold = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

print(f"\n🔄 {N_SPLITS}-Fold 교차 검증 시작...")

for fold, (train_idx, val_idx) in enumerate(kfold.split(X_np, y)):
    print(f"\n--- Fold {fold+1}/{N_SPLITS} ---")
    
    # XGBoost, LightGBM용 numpy
    X_tr, X_va = X_np[train_idx], X_np[val_idx]
    y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
    
    # CatBoost용 DataFrame
    X_tr_df = X_df.iloc[train_idx]
    X_va_df = X_df.iloc[val_idx]
    
    # XGBoost
    xgb_model = XGBClassifier(
        n_estimators=500,
        max_depth=4,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        random_state=SEED,
        verbosity=0,
        use_label_encoder=False,
        eval_metric='error'
    )
    xgb_model.fit(X_tr, y_tr)
    oof_xgb[val_idx] = xgb_model.predict_proba(X_va)[:, 1]
    test_xgb += xgb_model.predict_proba(X_test_np)[:, 1] / N_SPLITS
    
    # LightGBM
    lgb_model = LGBMClassifier(
        n_estimators=500,
        max_depth=5,
        learning_rate=0.03,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        class_weight='balanced',
        random_state=SEED,
        verbose=-1
    )
    lgb_model.fit(X_tr, y_tr)
    oof_lgb[val_idx] = lgb_model.predict_proba(X_va)[:, 1]
    test_lgb += lgb_model.predict_proba(X_test_np)[:, 1] / N_SPLITS
    
    # CatBoost (⭐ DataFrame + 범주형 컬럼명으로 처리!)
    cat_model = CatBoostClassifier(
        n_estimators=500,
        max_depth=5,
        learning_rate=0.03,
        auto_class_weights='Balanced',
        cat_features=cat_features_names,  # ⭐ 컬럼명으로 지정!
        random_state=SEED,
        verbose=0
    )
    cat_model.fit(X_tr_df, y_tr)  # DataFrame 전달
    oof_cat[val_idx] = cat_model.predict_proba(X_va_df)[:, 1]
    test_cat += cat_model.predict_proba(X_test_df)[:, 1] / N_SPLITS
    
    # Fold별 성능
    xgb_acc = accuracy_score(y_va, (oof_xgb[val_idx] >= 0.5).astype(int))
    lgb_acc = accuracy_score(y_va, (oof_lgb[val_idx] >= 0.5).astype(int))
    cat_acc = accuracy_score(y_va, (oof_cat[val_idx] >= 0.5).astype(int))
    print(f"   XGB: {xgb_acc:.5f}, LGB: {lgb_acc:.5f}, CAT: {cat_acc:.5f}")

print("\n✅ K-Fold OOF 완료!")

In [ ]:
# OOF 전체 성능
print("\n📊 OOF 전체 성능 (threshold=0.5):")
print(f"   XGBoost:  {accuracy_score(y, (oof_xgb >= 0.5).astype(int)):.5f}")
print(f"   LightGBM: {accuracy_score(y, (oof_lgb >= 0.5).astype(int)):.5f}")
print(f"   CatBoost: {accuracy_score(y, (oof_cat >= 0.5).astype(int)):.5f}")

## 📌 STEP 7: ⭐ OOF 기반 가중치/임계값 최적화

In [ ]:
print("=" * 60)
print("🔍 OOF 기반 가중치 최적화")
print("=" * 60)

best_score = 0
best_weights = (0.33, 0.33, 0.34)

# 가중치 그리드 서치 (0.05 단위)
for w1 in np.arange(0.2, 0.6, 0.05):
    for w2 in np.arange(0.2, 0.6, 0.05):
        w3 = round(1 - w1 - w2, 2)
        if w3 >= 0.1:  # 최소 10%
            oof_blend = w1 * oof_xgb + w2 * oof_lgb + w3 * oof_cat
            pred = (oof_blend >= 0.5).astype(int)
            acc = accuracy_score(y, pred)
            if acc > best_score:
                best_score = acc
                best_weights = (w1, w2, w3)

w_xgb, w_lgb, w_cat = best_weights
print(f"\n🏆 최적 가중치: XGB={w_xgb:.2f}, LGB={w_lgb:.2f}, CAT={w_cat:.2f}")
print(f"   OOF Accuracy (threshold=0.5): {best_score:.5f}")

In [ ]:
print("\n" + "=" * 60)
print("🔍 OOF 기반 임계값 최적화 (0.01 단위)")
print("=" * 60)

# 최적 가중치로 블렌딩
oof_final = w_xgb * oof_xgb + w_lgb * oof_lgb + w_cat * oof_cat

best_threshold = 0.5
best_acc = 0
results = []

for thresh in np.arange(0.35, 0.65, 0.01):
    pred = (oof_final >= thresh).astype(int)
    acc = accuracy_score(y, pred)
    f1 = f1_score(y, pred)
    results.append({'threshold': thresh, 'accuracy': acc, 'f1': f1})
    if acc > best_acc:
        best_acc = acc
        best_threshold = thresh

results_df = pd.DataFrame(results)
print("\n상위 10개 임계값:")
print(results_df.nlargest(10, 'accuracy').to_string(index=False))

print(f"\n🏆 최적 임계값: {best_threshold:.2f}")
print(f"   OOF Accuracy: {best_acc:.5f}")

## 📌 STEP 8: 최종 예측

In [ ]:
# Test 예측 블렌딩
test_final = w_xgb * test_xgb + w_lgb * test_lgb + w_cat * test_cat

# 최적 임계값 적용
final_prediction = (test_final >= best_threshold).astype(int)

print(f"🎯 최종 설정:")
print(f"   가중치: XGB={w_xgb:.2f}, LGB={w_lgb:.2f}, CAT={w_cat:.2f}")
print(f"   임계값: {best_threshold:.2f}")
print(f"\n예측 결과 분포:")
print(f"   0 (비흡연): {(final_prediction == 0).sum()}명 ({(final_prediction == 0).mean()*100:.1f}%)")
print(f"   1 (흡연):   {(final_prediction == 1).sum()}명 ({(final_prediction == 1).mean()*100:.1f}%)")

## 📌 STEP 9: 제출 파일 생성

In [ ]:
submission_df = submission.copy()
submission_df['label'] = final_prediction
submission_df['label'] = submission_df['label'].astype(int)

print("📋 제출 파일 미리보기:")
display(submission_df.head(10))

# 저장
output_path = result_path + 'submission_v7_stable.csv'
submission_df.to_csv(output_path, index=False)
print(f"\n✅ 저장: {output_path}")

In [ ]:
# 검증
print("🔍 제출 파일 검증:")
print(f"   행: {len(submission_df)}")
print(f"   타입: {submission_df['label'].dtype}")
print(f"   값: {sorted(submission_df['label'].unique())}")

if submission_df['label'].dtype in ['int64','int32'] and set(submission_df['label'].unique()).issubset({0,1}):
    print("\n✅ 검증 통과!")
else:
    print("\n⚠️ 검증 실패!")

## 📌 STEP 10: 다운로드

In [ ]:
from google.colab import files
files.download(output_path)

print("\n" + "=" * 60)
print("🎉 V7 안정화 버전 완료!")
print("=" * 60)
print(f"\n📊 V7 핵심 변경사항 (vs V6):")
print(f"   ✅ OOF 기반 튜닝 (단일 홀드아웃 X)")
print(f"   ✅ CatBoost 범주형 처리: {cat_cols_updated}")
print(f"   ✅ 파생 피처 최소화: 4개만")
print(f"   ✅ 이상치 클리핑 OFF")
print(f"   ✅ 모델 단순화: XGB, LGB, CAT (RF 제외)")
print(f"\n📊 최종 설정:")
print(f"   가중치: XGB={w_xgb:.2f}, LGB={w_lgb:.2f}, CAT={w_cat:.2f}")
print(f"   임계값: {best_threshold:.2f}")
print(f"   OOF Accuracy: {best_acc:.5f}")
print(f"\n🚀 제출하세요!")